# Подтипы скама: multilabel-голова

Три класса: `phishing_link`, `money_request`, `other`. Мультилейбл — не
взаимоисключающие.

**Протокол тот же, что у бинарной задачи.** Обучение на train 1392 (silver-метки
DeepSeek), отбор чекпоинта по **macro-F1 подтипов на gold-dev 100** (человеческие
метки), финальный замер на gold-test 303. Разбиение `gold_dev_split.json`
переиспользуется, с ассертами. Сиды 20260721/22/23, lr 2e-5, wd 0.01, warmup 0.2,
max_length 512, 5 эпох, patience=2.

⚠ **Результат заведомо будет плохим, и это ожидаемо.** Согласие silver с gold по
подтипам: macro-κ 0.230, полное совпадение множества 45.4%. Обучающий сигнал
плохо определён. Цель — измерить, а не вытянуть.

Дополнительный фактор: сдвиг априорных частот. В silver-train `phishing_link`
встречается у 257 из 1392 (18%), а в gold-test — у 133 из 303 (44%).

Снимки предсказаний на gold-dev И gold-test сохраняются на КАЖДОЙ эпохе, как в
`epoch_cost`. gold_test добавлен третьим eval-набором; на отбор он не влияет —
`metric_for_best_model = eval_gold_dev_macro_f1`.

**Kaggle:** GPU T4, Internet On, датасет `scam-onset-v1`.

## Конфигурация и данные

In [ ]:
import os, sys, json, csv, glob, pathlib, time, copy

def find_one(name):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    assert hits, f"не найден {name} под /kaggle/input"
    return hits[0]

CORPUS   = find_one("corpus_v1.jsonl")
GOLD     = find_one("gold_test_v1.csv")
GOLD_DEV = find_one("gold_dev_split.json")

OUT = pathlib.Path("/kaggle/working/subtypes"); OUT.mkdir(parents=True, exist_ok=True)
TB_ROOT = pathlib.Path("/kaggle/working/tb")

SUBTYPES = ["phishing_link", "money_request", "other"]
MODEL_NAME = "xlm-roberta-base"
SEEDS = [20260721, 20260722, 20260723]
LR, WEIGHT_DECAY, WARMUP_RATIO = 2e-5, 0.01, 0.2
EPOCHS, MAX_LENGTH, BATCH_SIZE, PATIENCE = 5, 512, 16, 2

import torch, transformers
DEVICE, USE_FP16 = ("cuda", True) if torch.cuda.is_available() else ("cpu", False)
print("torch", torch.__version__, "| transformers", transformers.__version__)
print("устройство:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "",
      "| GPU:", torch.cuda.device_count() if DEVICE == "cuda" else 0)

In [ ]:
def read_corpus(path):
    out = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line: out.append(json.loads(line))
    return out

def build_text(rec):
    return "\n".join(f"[{t['i']}|{t['speaker']}] {t['text']}" for t in rec["turns"])

def read_gold_csv(path):
    """dialog_id → {is_scam, subs}. Файл только читается."""
    out = {}
    with open(path, encoding="utf-8-sig", newline="") as f:
        for r in csv.DictReader(f):
            did = (r.get("dialog_id") or "").strip()
            if did:
                out[did] = {"is_scam": int(r["is_scam"]),
                            "subs": [s.strip() for s in (r.get("subtypes") or "").split("|") if s.strip()]}
    return out

def multihot(subs):
    return [1.0 if s in subs else 0.0 for s in SUBTYPES]

records = read_corpus(CORPUS)
gold    = read_gold_csv(GOLD)

train_recs = [r for r in records if r.get("split") == "train"]
val_recs   = [r for r in records if r.get("split") == "val"]
for r in train_recs + val_recs:
    r["y"] = multihot(r.get("subtypes") or [])       # silver

test_split = [r for r in records if r.get("split") == "test"]
all_test = []
for r in test_split:
    r2 = {k: v for k, v in r.items() if k not in ("is_scam", "subtypes")}   # silver-метки выбрасываем
    if r2["id"] in gold:
        r2["y"] = multihot(gold[r2["id"]]["subs"])   # gold
        r2["y_silver"] = multihot(r.get("subtypes") or [])
        r2["gold_is_scam"] = gold[r2["id"]]["is_scam"]
        all_test.append(r2)
assert len(all_test) == len(gold), f"джойн неполный: {len(all_test)} из {len(gold)}"

sm = json.load(open(GOLD_DEV, encoding="utf-8"))
assign = sm["assignment"]
dev_recs  = [r for r in all_test if assign.get(r["id"]) == "dev"]
test_recs = [r for r in all_test if assign.get(r["id"]) != "dev"]
assert len(dev_recs) == sm["n_dev"] and len(test_recs) == sm["n_test"], "разбиение не совпало"
gd = {r.get("scenario_group") for r in dev_recs}
gt = {r.get("scenario_group") for r in test_recs}
assert not (gd & gt), "группы scenario_group разорваны между dev и test"
assert len({len(dev_recs), len(val_recs), len(test_recs)}) == 3, "размеры наборов совпали"

print(f"разбиение gold: seed={sm['seed']} dev={len(dev_recs)} test={len(test_recs)}, "
      f"групп {len(gd)}/{len(gt)}, пересечение 0")
print(f"train {len(train_recs)} (silver) / val {len(val_recs)} (silver) / "
      f"gold-dev {len(dev_recs)} / gold-test {len(test_recs)}")
import numpy as np
for nm, rs in (("train silver", train_recs), ("gold-dev", dev_recs), ("gold-test", test_recs)):
    y = np.array([r["y"] for r in rs])
    print(f"  {nm:12s}: " + ", ".join(f"{s}={int(y[:, i].sum())}" for i, s in enumerate(SUBTYPES)) +
          f", без подтипов={int((y.sum(1) == 0).sum())}")

## Обучение

In [ ]:
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          EarlyStoppingCallback, Trainer, TrainingArguments, set_seed)
from sklearn.metrics import f1_score

tok = AutoTokenizer.from_pretrained(MODEL_NAME)

class MLDataset(torch.utils.data.Dataset):
    def __init__(self, recs):
        self.enc = tok([build_text(r) for r in recs], truncation=True,
                       max_length=MAX_LENGTH, padding="max_length")
        self.labels = [r["y"] for r in recs]
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[i], dtype=torch.float)
        return item

ds_train = MLDataset(train_recs); ds_val = MLDataset(val_recs)
ds_dev = MLDataset(dev_recs);     ds_test = MLDataset(test_recs)
print("токенизация готова")

def ml_metrics(y_true, y_pred):
    """macro/micro-F1 + per-label F1. y_* — бинарные матрицы (n, 3)."""
    out = {"macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
           "micro_f1": float(f1_score(y_true, y_pred, average="micro", zero_division=0))}
    for i, s in enumerate(SUBTYPES):
        out[f"f1_{s}"] = float(f1_score(y_true[:, i], y_pred[:, i], zero_division=0))
    return out

def run_seed(seed):
    set_seed(seed); t0 = time.time()
    tb_dir = TB_ROOT / f"subtypes_{MODEL_NAME.split('/')[-1]}_seed{seed}"
    tb_dir.mkdir(parents=True, exist_ok=True)
    os.environ["TENSORBOARD_LOGGING_DIR"] = str(tb_dir)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(SUBTYPES),
        problem_type="multi_label_classification", torch_dtype=torch.float32)

    dev_snaps, test_snaps = [], []
    capture = {"on": True}

    def compute_metrics(ep):
        logits, labels = ep
        probs = torch.sigmoid(torch.tensor(logits)).numpy()
        preds = (probs >= 0.5).astype(int)
        labels = np.asarray(labels).astype(int)
        if capture["on"] and len(labels) in (len(dev_recs), len(test_recs)):
            snap = {"y_true": labels.tolist(), "y_pred": preds.tolist(),
                    "probs": probs.tolist()}
            (dev_snaps if len(labels) == len(dev_recs) else test_snaps).append(snap)
        return ml_metrics(labels, preds)

        # ФИНАЛЬНАЯ модель: веса обязаны пережить сессию. save_total_limit=2 и
        # обязательный trainer.save_model(...) наружу — см. CLAUDE.md,
        # «Конвенции: сохранение моделей».
    targs = TrainingArguments(
        output_dir=str(OUT / f"ckpt_seed{seed}"),
        learning_rate=LR, weight_decay=WEIGHT_DECAY, warmup_ratio=WARMUP_RATIO,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
        seed=seed, data_seed=seed, fp16=USE_FP16,
        eval_strategy="epoch", save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_gold_dev_macro_f1",   # отбор ТОЛЬКО по gold-dev
        greater_is_better=True, save_total_limit=2, logging_steps=10,
        report_to="tensorboard", logging_dir=str(tb_dir),
    )
    trainer = Trainer(model=model, args=targs, train_dataset=ds_train,
                      eval_dataset={"silver_val": ds_val, "gold_dev": ds_dev,
                                    "gold_test": ds_test},
                      processing_class=tok, compute_metrics=compute_metrics,
                      callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)])
    trainer.train()
    hist = copy.deepcopy(trainer.state.log_history)

    gv = [h for h in hist if "eval_gold_dev_macro_f1" in h]
    tr = [h for h in hist if "loss" in h and "eval_gold_dev_loss" not in h
          and "eval_silver_val_loss" not in h and "eval_gold_test_loss" not in h]
    per_epoch = []
    for h in gv:
        ep = h["epoch"]
        v = [x["loss"] for x in tr if ep - 1 < x["epoch"] <= ep]
        rec = {"epoch": int(round(ep)), "seed": seed,
               "train_loss": float(sum(v) / len(v)) if v else None,
               "gold_dev_macro_f1": float(h["eval_gold_dev_macro_f1"])}
        for src, pfx in (("eval_silver_val", "silver_val"), ("eval_gold_test", "gold_test")):
            g = next((x for x in hist if f"{src}_macro_f1" in x and x["epoch"] == ep), None)
            if g:
                rec[f"{pfx}_macro_f1"] = float(g[f"{src}_macro_f1"])
        per_epoch.append(rec)
    best_epoch = max(per_epoch, key=lambda x: x["gold_dev_macro_f1"])["epoch"]

    assert len(dev_snaps) == len(per_epoch) == len(test_snaps), \
        f"эпох {len(per_epoch)}, снимков dev {len(dev_snaps)}, test {len(test_snaps)}"
    for tag, snaps, recs in (("dev", dev_snaps, dev_recs), ("test", test_snaps, test_recs)):
        json.dump({"seed": seed, "subtypes": SUBTYPES,
                   "dialog_ids": [r["id"] for r in recs],
                   "langs": [r.get("lang") for r in recs],
                   "y_silver": [r.get("y_silver") for r in recs],
                   "epochs": [p["epoch"] for p in per_epoch], "per_epoch": snaps},
                  open(OUT / f"gold_{tag}_per_epoch_seed{seed}.json", "w", encoding="utf-8"),
                  ensure_ascii=False)
    json.dump(hist, open(OUT / f"history_seed{seed}.json", "w", encoding="utf-8"),
              ensure_ascii=False, indent=2)

    capture["on"] = False
    po = trainer.predict(ds_test)
    probs = torch.sigmoid(torch.tensor(po.predictions)).numpy()
    res = {"seed": seed, "per_epoch": per_epoch, "best_epoch": best_epoch,
           "epochs_ran": len(per_epoch),
           "best_checkpoint": trainer.state.best_model_checkpoint,
           "best_gold_dev_macro_f1": float(trainer.state.best_metric),
           "minutes": round((time.time() - t0) / 60, 1),
           "final_probs": probs.tolist()}
    del trainer, model; torch.cuda.empty_cache()
    print(f"[seed {seed}] {res['minutes']} мин | эпох {res['epochs_ran']} | "
          f"лучшая эпоха {best_epoch} | gold-dev macro-F1 {res['best_gold_dev_macro_f1']:.4f}")
    return res

ALL = []
for s in SEEDS:
    print("=" * 74); print(f"СИД {s}"); print("=" * 74)
    ALL.append(run_seed(s))

## Ход по эпохам и сохранение

In [ ]:
print(f"{'сид':>10} {'эпоха':>6} {'train loss':>11} {'silver_val':>11} {'gold_dev':>10} {'gold_test':>10}")
print("-" * 62)
for r in ALL:
    for p in r["per_epoch"]:
        mark = " ←" if p["epoch"] == r["best_epoch"] else ""
        tl = f"{p['train_loss']:.4f}" if p["train_loss"] is not None else "   —   "
        print(f"{p['seed']:>10} {p['epoch']:>6} {tl:>11} "
              f"{p.get('silver_val_macro_f1', float('nan')):>11.4f} "
              f"{p['gold_dev_macro_f1']:>10.4f} "
              f"{p.get('gold_test_macro_f1', float('nan')):>10.4f}{mark}")
    print("-" * 62)
print("← лучшая эпоха по macro-F1 на gold-dev")
print("лучшие эпохи:", [r["best_epoch"] for r in ALL],
      "| эпох отработано:", [r["epochs_ran"] for r in ALL], f"из {EPOCHS}")

json.dump({"model": MODEL_NAME, "subtypes": SUBTYPES, "seeds": SEEDS,
           "config": {"lr": LR, "weight_decay": WEIGHT_DECAY, "warmup_ratio": WARMUP_RATIO,
                      "epochs": EPOCHS, "batch_size": BATCH_SIZE, "max_length": MAX_LENGTH,
                      "patience": PATIENCE, "selection_metric": "eval_gold_dev_macro_f1",
                      "problem_type": "multi_label_classification",
                      "device": DEVICE, "n_gpu": torch.cuda.device_count() if DEVICE == "cuda" else 0,
                      "versions": {"torch": torch.__version__,
                                   "transformers": transformers.__version__,
                                   "cuda": torch.version.cuda,
                                   "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else None}},
           "split": {"seed": sm["seed"], "n_dev": len(dev_recs), "n_test": len(test_recs)},
           "runs": [{k: v for k, v in r.items() if k != "final_probs"} for r in ALL],
           "best_epochs": [r["best_epoch"] for r in ALL],
           "epochs_ran": [r["epochs_ran"] for r in ALL]},
          open(OUT / "results_raw.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("\nсохранено:", sorted(p.name for p in OUT.glob("*.json")))